# Week 2 — Baseline model for electrode voltage
**Goal:** predict the average voltage of an insertion electrode from its chemistry, and measure honestly how good the prediction is.

**This notebook:**
1. Loads `electrodes_clean.csv` from your Google Drive (saved in Week 1)
2. Builds physics-based features for the **host framework** and the **working ion**
3. Splits the data **by framework**, so the test set only contains host materials the model has never seen
4. Trains two reference models and a gradient-boosting model (XGBoost)
5. Evaluates errors overall and per ion, makes a parity plot and shows which features matter most
6. Saves the model, features and predictions back to Drive

Runs on Colab's normal CPU runtime (no GPU needed). Run cells top to bottom.

In [ ]:
# 1. Install packages and connect Google Drive
!pip -q install pymatgen xgboost
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"

In [ ]:
import ast, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

df = pd.read_csv(f"{DATA_DIR}/electrodes_clean.csv")
df["elements"] = df["elements"].apply(ast.literal_eval)

# Focus ions (enough data). Y, Rb, Al, Cs are kept aside and reported as a limitation.
ION_ORDER = ["Li", "Na", "K", "Mg", "Ca", "Zn"]
PALETTE = dict(zip(ION_ORDER, sns.color_palette("tab10", len(ION_ORDER))))   # one fixed colour per ion
df = df[df["working_ion"].isin(ION_ORDER)].reset_index(drop=True)
print(f"{len(df)} electrodes for the focus ions")
print(df["working_ion"].value_counts().reindex(ION_ORDER))

## 2. Features
A model cannot read a chemical formula, so we turn each electrode into numbers that carry physics:

**Host framework** (from `framework_formula`, e.g. FePO4F): composition-weighted **mean, standard deviation, min and max** of element properties — electronegativity, atomic radius, atomic number, mass, row, group, Mendeleev number and first ionization energy. Plus the fraction of transition metals, the anion type (O, F, S, Se, N, Cl, Br) and polyanion formers (P, S, Si, B, C, Mo, W with O), which raise the voltage through the *inductive effect*.

**Working ion:** its charge (1 for Li/Na/K, 2 for Mg/Ca/Zn), ionic radius, electronegativity, ionization energy and mass, and how much ion is inserted per framework atom (`fracA_discharge`).

We deliberately do **not** use other DFT outputs such as volume change or capacity steps, because they would not be known for a brand-new material.

In [ ]:
from pymatgen.core import Composition, Element

PROPS = {
    "X": lambda e: e.X,
    "radius": lambda e: float(e.atomic_radius) if e.atomic_radius is not None else np.nan,
    "Z": lambda e: e.Z,
    "mass": lambda e: float(e.atomic_mass),
    "row": lambda e: e.row,
    "group": lambda e: e.group,
    "mendeleev": lambda e: e.mendeleev_no,
    "ionization": lambda e: e.ionization_energy,
}
_cache = {}
def eprop(sym, name):
    key = (sym, name)
    if key not in _cache:
        try:
            v = PROPS[name](Element(sym)); _cache[key] = np.nan if v is None else float(v)
        except Exception:
            _cache[key] = np.nan
    return _cache[key]

TM = {str(e) for e in Element if e.is_transition_metal}
ANIONS = ["O", "F", "S", "Se", "N", "Cl", "Br"]
POLY = ["P", "Si", "B", "C", "Mo", "W"]

def host_features(formula):
    comp = Composition(formula).fractional_composition
    syms = [str(e) for e in comp.elements]; w = np.array([comp[e] for e in comp.elements])
    f = {}
    for name in PROPS:
        v = np.array([eprop(s, name) for s in syms])
        m = ~np.isnan(v)
        if m.any():
            ww = w[m] / w[m].sum(); vv = v[m]
            mean = (ww * vv).sum()
            f[f"host_{name}_mean"] = mean
            f[f"host_{name}_std"] = np.sqrt((ww * (vv - mean) ** 2).sum())
            f[f"host_{name}_min"], f[f"host_{name}_max"] = vv.min(), vv.max()
        else:
            for s in ("mean", "std", "min", "max"): f[f"host_{name}_{s}"] = np.nan
    f["host_n_elements"] = len(syms)
    f["host_frac_TM"] = sum(comp[e] for e in comp.elements if str(e) in TM)
    for a in ANIONS: f[f"host_frac_{a}"] = comp[a] if a in syms else 0.0
    has_O = "O" in syms
    f["host_polyanion"] = float(has_O and any(p in syms for p in POLY))
    tms = [s for s in syms if s in TM]
    f["host_TM_X_mean"] = np.mean([eprop(s, "X") for s in tms]) if tms else np.nan
    f["host_TM_Z_mean"] = np.mean([eprop(s, "Z") for s in tms]) if tms else np.nan
    return f

ION_CHARGE = {"Li": 1, "Na": 1, "K": 1, "Mg": 2, "Ca": 2, "Zn": 2}
def ion_features(ion):
    e = Element(ion); q = ION_CHARGE[ion]
    return {"ion_charge": q, "ion_radius": float(e.ionic_radii.get(q, np.nan)),
            "ion_X": e.X, "ion_ionization": e.ionization_energy, "ion_mass": float(e.atomic_mass)}

host = pd.DataFrame([host_features(f) for f in df["framework_formula"]])
ion = pd.DataFrame([ion_features(i) for i in df["working_ion"]])
X = pd.concat([host, ion], axis=1)
X["frac_ion_discharge"] = df["fracA_discharge"].values
y = df["average_voltage"].values
groups = df["framework_formula"].values
print("Feature matrix:", X.shape)
X.head()

## 3. Train/test split by framework
The same framework (e.g. FePO4F) often appears with several ions. If we split randomly, the model could see FePO4F+Li in training and FePO4F+Na in testing, and look better than it really is.

So we keep **every framework entirely in either the training or the test set** (`GroupShuffleSplit`), and use the same idea for cross-validation (`GroupKFold`).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
ion_tr, ion_te = df["working_ion"].values[tr], df["working_ion"].values[te]
print(f"train: {len(tr)} electrodes | test: {len(te)} electrodes")
print("Shared frameworks between train and test:", len(set(groups[tr]) & set(groups[te])))

## 4. Models
Always compare against simple references, so we know whether the model learned anything:
- **Reference 1:** predict the overall mean voltage for everything
- **Reference 2:** predict the mean voltage of each ion (knows only the ion, nothing about the host)
- **XGBoost:** gradient-boosted trees on all features, tuned lightly with grouped cross-validation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def scores(y_true, y_pred):
    return {"MAE (V)": mean_absolute_error(y_true, y_pred),
            "RMSE (V)": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

results = {}
results["Mean voltage"] = scores(y_te, np.full_like(y_te, y_tr.mean()))
ion_means = pd.Series(y_tr).groupby(ion_tr).mean()
results["Mean per ion"] = scores(y_te, pd.Series(ion_te).map(ion_means).values)
pd.DataFrame(results).T.round(3)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, cross_val_score

params_grid = [
    dict(n_estimators=600, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.7),
    dict(n_estimators=800, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.7),
    dict(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6),
]
gkf = GroupKFold(n_splits=5)
best, best_mae = None, np.inf
for p in params_grid:
    m = XGBRegressor(**p, min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1)
    mae = -cross_val_score(m, X_tr, y_tr, groups=groups[tr], cv=gkf,
                           scoring="neg_mean_absolute_error").mean()
    print(f"depth={p['max_depth']}, trees={p['n_estimators']}: CV MAE = {mae:.3f} V")
    if mae < best_mae: best, best_mae = p, mae

model = XGBRegressor(**best, min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
results["XGBoost"] = scores(y_te, pred)
pd.DataFrame(results).T.round(3)

## 5. How good is it?

In [ ]:
per_ion = pd.DataFrame([{"ion": i, "n_test": int((ion_te == i).sum()), **scores(y_te[ion_te == i], pred[ion_te == i])}
                        for i in ION_ORDER if (ion_te == i).sum() > 5]).set_index("ion").round(3)
print("Test errors per ion:"); per_ion

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for i in ION_ORDER:
    m = ion_te == i
    ax.scatter(y_te[m], pred[m], s=14, alpha=0.7, label=i, color=PALETTE[i])
lim = [-0.2, 6.2]; ax.plot(lim, lim, "k--", lw=1)
ax.set(xlim=lim, ylim=lim, xlabel="DFT average voltage (V)", ylabel="Predicted voltage (V)",
       title=f"XGBoost test set  (MAE = {results['XGBoost']['MAE (V)']:.2f} V)")
ax.legend(title="Ion"); plt.tight_layout(); plt.savefig("fig_parity_xgb.png", dpi=200); plt.show()

In [ ]:
imp = pd.Series(model.get_booster().get_score(importance_type="gain")).sort_values(ascending=False).head(20)
imp = imp / imp.sum()
imp[::-1].plot.barh(figsize=(7, 6), color="#4C72B0", title="Top 20 features (share of total gain)")
plt.xlabel("Relative importance"); plt.tight_layout(); plt.savefig("fig_feature_importance.png", dpi=200); plt.show()

In [ ]:
# Where does the model fail? The 10 largest errors in the test set
err = pd.DataFrame({"framework": groups[te], "ion": ion_te, "DFT_V": y_te, "pred_V": pred})
err["abs_error"] = (err["DFT_V"] - err["pred_V"]).abs()
err.sort_values("abs_error", ascending=False).head(10).round(2)

## 6. Save everything to Drive

In [ ]:
import shutil, json
model.save_model("xgb_voltage_baseline.json")
X.assign(average_voltage=y, working_ion=df["working_ion"], framework=groups).to_csv("features_week2.csv", index=False)
err.to_csv("test_predictions_xgb.csv", index=False)
pd.DataFrame(results).T.to_csv("results_week2.csv")
per_ion.to_csv("results_per_ion_week2.csv")
json.dump({"best_params": best, "cv_mae": best_mae}, open("xgb_params.json", "w"), indent=2)
for f in ["xgb_voltage_baseline.json", "features_week2.csv", "test_predictions_xgb.csv", "results_week2.csv",
          "results_per_ion_week2.csv", "xgb_params.json", "fig_parity_xgb.png", "fig_feature_importance.png"]:
    shutil.copy(f, DATA_DIR)
print("Saved to", DATA_DIR)

## What to send me
1. The **results table** (Mean voltage / Mean per ion / XGBoost)
2. The **per-ion test errors**
3. The **parity plot** and the **feature-importance plot**

**How to read the results:** a typical MAE for this kind of composition-only model is roughly 0.3–0.6 V. What matters most is that XGBoost clearly beats "mean per ion" — that proves the model learned something about the host chemistry, not just the ion. In Week 3 we add the crystal structure with a graph neural network, and in Week 4 we run the Li → Na/K transfer test.